# Week 4 — Introduction to scikit-learn II
## From packing-time predictions to model selection

**Guiding question:** Before packing starts, can we rank orders by their risk of taking more than 15 minutes to pack?

This notebook is preparation material for independent study. Run the cells in order, read the explanations and pause at the self-checks. All examples include complete code. The orders are **simulated**, not evidence about a real warehouse. Their generating process is available in the project, but try investigating the data first.

### Learning goals

By the end, you will be able to turn a prediction question into two classes, explain a ROC curve and its AUC, prepare inputs for a model, and compare models using several training and validation splits. You will also see how to choose model settings systematically and assess the selected approach on orders kept aside for a final test. Each of these ideas is introduced below.

Keep asking: **What is our question? What is our baseline? How do we evaluate the result? What can we conclude?**

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th colspan="2">Contents</th></tr></thead>
<tbody>
<tr><td>1</td><td><a href="#section-1-load-and-understand-the-data">Load and understand the data</a></td></tr>
<tr><td>2</td><td><a href="#section-2-construct-the-target">Construct the target</a></td></tr>
<tr><td>3</td><td><a href="#section-3-reserve-a-test-set">Reserve a test set</a></td></tr>
<tr><td>4</td><td><a href="#section-4-scores-decisions-and-confusion-matrices">Scores, decisions and confusion matrices</a></td></tr>
<tr><td>5</td><td><a href="#section-5-roc-curve-and-roc-auc">ROC curve and ROC-AUC</a></td></tr>
<tr><td>6</td><td><a href="#section-6-scaling-and-feature-engineering">Scaling and feature engineering</a></td></tr>
<tr><td>7</td><td><a href="#section-7-pipelines-and-cross-validation">Pipelines and cross-validation</a></td></tr>
<tr><td>8</td><td><a href="#section-8-a-small-neural-network">A small neural network</a></td></tr>
<tr><td>9</td><td><a href="#section-9-search-hyperparameters-and-compare-four-candidates">Search hyperparameters and compare four candidates</a></td></tr>
<tr><td>10</td><td><a href="#section-10-investigate-scaling">Investigate scaling</a></td></tr>
<tr><td>11</td><td><a href="#section-11-select-and-evaluate-once-on-the-test-set">Select and evaluate once on the test set</a></td></tr>
<tr><td>12</td><td><a href="#section-12-recap-and-outlook">Recap and outlook</a></td></tr>
<tr><td>★</td><td><a href="#section-optional-your-turn">Optional: Your turn</a></td></tr>
</tbody>
</table>

<h2 id="section-1-load-and-understand-the-data">1. Load and understand the data</h2>

Week 3 used the same orders to predict packing time in minutes. This week we ask a different question about the same orders: which ones are likely to take more than 15 minutes? The notebook and the `data` folder are in the repository root. Run the cells in order so that later cells can use values created earlier.

We start by examining the orders available for our prediction task.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook and data folder are both in the project root.
data_path = Path("data/orders.csv")

# Read the file into a DataFrame and inspect its size and first rows.
orders = pd.read_csv(data_path)
print("Rows and columns:", orders.shape)
orders.head()

Each row represents one order. Its **features** are the pieces of information we may give a model as inputs. Here these are known before packing begins:

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th>Feature</th><th>Meaning</th></tr></thead>
<tbody>
<tr><td><code>item_count</code></td><td>Number of items in the order</td></tr>
<tr><td><code>fragile_count</code></td><td>Number of items that need careful handling</td></tr>
<tr><td><code>storage_zones</code></td><td>Number of warehouse areas from which items are collected</td></tr>
<tr><td><code>gift_wrap</code></td><td>Whether gift wrapping was requested: 1 for yes, 0 for no</td></tr>
<tr><td><code>total_weight_kg</code></td><td>Total weight in kilograms</td></tr>
<tr><td><code>order_value_eur</code></td><td>Order value in euros</td></tr>
<tr><td><code>package_side_cm</code></td><td>Side length in centimetres of the planned cubic package</td></tr>
</tbody>
</table>

Two other columns have different roles. `order_id` identifies a row; it does not describe the order. `packing_time_min` is observed **after** packing has finished. We can use it to construct the outcome we want to predict, but we cannot give it to a model that predicts before packing.

Inspect the column types, missing values and distributions before deciding how to use the data.

In [ ]:
# Check column types, missing values and repeated identifiers.
print(orders.dtypes)
print("\nMissing values:\n", orders.isna().sum())
print("\nDuplicate IDs:", orders["order_id"].duplicated().sum())

# Summarize numeric columns; round the displayed statistics.
orders.describe().round(2)

**Self-check:** Which column would give the answer away if used as a feature?

The observed packing time reveals which side of 15 minutes an order falls on. It is available for past orders but not for a new order before packing. Including it among the inputs would give the model information that would not be available when making a real prediction. This is an example of **data leakage**.

<h2 id="section-2-construct-the-target">2. Construct the target</h2>

In week 3 the goal was to predict a number of minutes. Here the goal is to decide which of two groups an order belongs to: this is **binary classification**. We call the outcome to predict the **target**.

We define a long order as one taking **more than** 15 minutes: class 1. An order taking 15 minutes or less belongs to class 0. The 15-minute boundary represents an example business decision, such as deciding which orders may need additional preparation. We fix this definition before comparing models. Let's see how common each class is.

In [ ]:
# The comparison produces True/False; astype(int) converts these to 1/0.
orders["over_15_min"] = (orders["packing_time_min"] > 15).astype(int)

# Count each class, then divide by the number of orders for its share.
counts = orders["over_15_min"].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "proportion": counts / len(orders)}))
counts.plot.bar(rot=0, title="Orders by class", xlabel="Over 15 minutes (1 = yes)", ylabel="Orders")
plt.show()

**Accuracy** is the fraction of predictions that are correct. Imagine 95 of 100 orders are short: always predicting ‘short’ would give 95% accuracy, while failing to identify any of the five long orders. When one class is much more frequent than the other, we call this **class imbalance**. Whether missing rare cases matters depends on the application.

The next cell explores how the class proportions would change with different definitions of a long order. Our actual target remains fixed at 15 minutes.

**Self-check:** If only 5 of 100 orders are long, what accuracy would a model achieve by predicting every order as short? How many long orders would it identify?

It would be correct on 95 orders, giving 95% accuracy, but it would identify none of the long orders. We therefore need to look beyond accuracy when judging whether a model is useful.

In [ ]:
# This loop changes only the illustration; it does not change our target.
for threshold in [15, 25, 35]:
    print(f"Over {threshold} minutes: {(orders['packing_time_min'] > threshold).mean():.1%}")

<h2 id="section-3-reserve-a-test-set">3. Reserve a test set</h2>

Before comparing models, we reserve 20% of the orders as a **test set**. These orders are set aside until the end, so they can provide a check on choices made using the other 80%. We call those remaining orders the **training data**. The split keeps approximately the same proportion of long orders in each group.

This is a random split, suitable for illustrating our independently simulated orders. For real orders arriving over time, a split based on past and future may be more appropriate. We exclude the identifier and the observed packing time from the model inputs.

As in Week 3, the orders used for developing our approach can be called **development data**. Here, the variable `X_train` holds the development inputs after the test split. In each cross-validation run below, only part of these rows fits the model; the remaining part acts as validation data. The separate test set does not take part in these choices.

In [ ]:
from sklearn.model_selection import train_test_split

# X contains inputs available before packing; y contains the target.
features = ["item_count", "fragile_count", "storage_zones", "gift_wrap",
            "total_weight_kg", "order_value_eur", "package_side_cm"]
X = orders[features].copy()
y = orders["over_15_min"].copy()

# Reserve 20% for testing; stratify approximately preserves class proportions.
# random_state makes this particular split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train/test sizes:", len(X_train), len(X_test))
print("Positive proportion — train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3))

<h2 id="section-4-scores-decisions-and-confusion-matrices">4. Scores, decisions and confusion matrices</h2>

A model can give each order a **score**: a higher score indicates stronger evidence that it is long. Our models will provide estimated probabilities between 0 and 1; a score of 0.8 is an estimate, not a guarantee that the order will be long.

To make a yes/no decision, we choose a **threshold**. At a threshold of 0.5, a score of 0.8 is classified as long and a score of 0.3 as short. Comparing predictions with actual outcomes gives four possibilities:

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th>Actual order</th><th>Model prediction</th><th>Name</th></tr></thead>
<tbody>
<tr><td>Long</td><td>Long</td><td>True positive (TP)</td></tr>
<tr><td>Short</td><td>Long</td><td>False positive (FP)</td></tr>
<tr><td>Short</td><td>Short</td><td>True negative (TN)</td></tr>
<tr><td>Long</td><td>Short</td><td>False negative (FN)</td></tr>
</tbody>
</table>

A **confusion matrix** counts these outcomes. The consequences of the two kinds of mistake depend on the decision: unnecessary preparation and an unexpected delay can have different costs.

This tiny example is independent of our training and test data.

In [ ]:
from sklearn.metrics import confusion_matrix
# Six known outcomes and example scores, in matching order.
example_y = np.array([0, 1, 0, 1, 1, 0])
example_scores = np.array([0.10, 0.80, 0.40, 0.60, 0.30, 0.70])

# Apply each threshold, then unpack the four confusion-matrix counts.
for threshold in [0.5, 0.3]:
    predictions = (example_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(example_y, predictions, labels=[0, 1]).ravel()
    print(f"Threshold {threshold}: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

**Self-check:** Compare the two thresholds in the output above. How did lowering the threshold change the number of orders predicted long? Did it reduce false positives?

At 0.5, three orders are flagged as long; at 0.3, five are flagged. The lower threshold catches the previously missed long order, but also incorrectly flags one additional short order. This is a trade-off, not a guarantee that every threshold change will affect both counts.

<h2 id="section-5-roc-curve-and-roc-auc">5. ROC curve and ROC-AUC</h2>

The **true positive rate** (TPR, also called recall) is `TP / (TP + FN)`: among actual long orders, the fraction identified as long. The **false positive rate** (FPR) is `FP / (FP + TN)`: among actual short orders, the fraction incorrectly flagged as long. Their denominators are different: one considers only long orders and the other only short orders. For example, if we flag 8 of 10 long orders, TPR is 0.8; if we also flag 2 of 10 short orders, FPR is 0.2.

The **ROC curve** (receiver operating characteristic curve) plots FPR on the horizontal axis against TPR on the vertical axis as we vary the decision threshold. A high threshold usually gives few positive predictions; lowering it gives more. Each point corresponds to a threshold. A point closer to the upper left means a high TPR and a low FPR at that threshold. Let's calculate several points from our example.

In [ ]:
# Calculate a point on the ROC curve for each chosen threshold.
for threshold in [1.0, 0.7, 0.5, 0.3, 0.0]:
    pred = (example_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(example_y, pred, labels=[0, 1]).ravel()
    print(f"threshold={threshold:.1f}  FPR={fp/(fp+tn):.2f}  TPR={tp/(tp+fn):.2f}")

`roc_curve` calculates points across the score thresholds. ROC-AUC is the **area under this curve**. It summarizes ranking across thresholds rather than judging one chosen threshold.

Imagine randomly choosing one long and one short order. An AUC of 0.8 means that, over such pairs, the long order receives the higher score about 80% of the time, counting tied scores as half a correct ranking. This is not the percentage of orders classified correctly. A constant score cannot distinguish orders and has an AUC of 0.5; perfect ranking has an AUC of 1.0. An AUC below 0.5 means the scores rank this sample in the opposite direction more often than not.

In the plot, the dashed diagonal is a reference: a model that assigns all orders the same score has AUC 0.5. The step-shaped example curve moves whenever the threshold passes one of the observed scores. Its appearance depends on this very small set of six examples.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
# Compute the curve and its area from the true labels and scores.
fpr, tpr, thresholds = roc_curve(example_y, example_scores)
example_auc = roc_auc_score(example_y, example_scores)

# Plot the model curve and the diagonal constant-score reference.
plt.plot(fpr, tpr, marker="o", label=f"Example AUC = {example_auc:.2f}")
plt.plot([0, 1], [0, 1], "--", label="Constant-score reference")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Example ROC curve")
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.show()
print(pd.DataFrame({"threshold": thresholds, "FPR": fpr, "TPR": tpr}))

**Self-check:** If we reverse every score, what happens to the ranking? Why does a good AUC not tell us which threshold to use?

AUC tells us how well the scores **rank** orders: are long orders usually given higher scores than short orders? It does not answer every question about those scores.

For example, suppose a model assigns a 0.8 score to many orders. If we interpret 0.8 as an 80% probability, we might expect roughly 80% of those orders to be long. Checking whether predicted probabilities match observed frequencies is called **calibration**. A model can rank orders well while its probabilities are too high or too low.

AUC also does not tell us which score threshold to choose. Flagging a short order may waste preparation time; failing to flag a long order may cause a delay. To choose a threshold, we would need to consider what these mistakes cost in our application.

If long orders are very rare, we may also want to know: of the orders we flag as long, how many actually are long? This fraction is called **precision**. Recall, introduced above as the true positive rate, is the fraction of all actual long orders that we flag. A precision-recall curve shows how these two fractions change as the threshold changes.

Finally, ROC-AUC needs **both actual classes** in the data being evaluated: without long orders there is no positive group to compare, and without short orders there is no negative group.

<h2 id="section-6-scaling-and-feature-engineering">6. Scaling and feature engineering</h2>

Different columns use different units: item counts, euros and cubic centimetres can have very different magnitudes. `StandardScaler` learns a mean and standard deviation **for each feature** and transforms a value to `(value - mean) / standard deviation`. Training values have mean approximately 0 and standard deviation 1; this does not make them normally distributed. Bringing features to comparable scales can help the training algorithm and influences how strongly a penalty on model weights affects each feature.

For example, if training orders have a mean weight of 10 kg and a standard deviation of 5 kg, an order weighing 15 kg becomes `(15 - 10) / 5 = 1`. The value 1 means one training standard deviation above the training mean. The scaler stores the training mean and standard deviation so that it can transform later orders in exactly the same way, even one order at a time.

The next small example is separate from our model data. In the actual comparison, the scaler learns its values separately within each training fold.

**Self-check:** If a new order weighs 20 kg and the stored training mean and standard deviation are 10 kg and 5 kg, what is its scaled value? It is 2: `(20 - 10) / 5`. We do not calculate a new mean from that one order.

In [ ]:
from sklearn.preprocessing import StandardScaler
# Two-dimensional input: three observations and one feature.
example_values = np.array([[10.0], [20.0], [30.0]])
demo_scaler = StandardScaler()
print("Original:", example_values.ravel())

# fit_transform learns the mean and standard deviation and applies them.
print("Standardized:", demo_scaler.fit_transform(example_values).ravel().round(3))

**Feature engineering** means creating additional model inputs from information we already have. For example, a cube with sides of 10 cm has volume `10 × 10 × 10 = 1,000 cm³`. The derived value does not require a new measurement; it presents the existing side length in a different form.

The six candidate features below express different hypotheses about packing. They may help a model, make no meaningful difference, or make its predictions worse. `item_count` and `storage_zones` are positive in this dataset, so the divisions are defined.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th>Option</th><th>Calculation</th><th>What it describes</th></tr></thead>
<tbody>
<tr><td><code>package_volume</code></td><td>side length cubed</td><td>Volume of the planned cubic box</td></tr>
<tr><td><code>average_weight</code></td><td>total weight ÷ items</td><td>Average weight per item</td></tr>
<tr><td><code>average_price</code></td><td>order value ÷ items</td><td>Average order value per item</td></tr>
<tr><td><code>fragile_share</code></td><td>fragile items ÷ items</td><td>Fraction of items that are fragile</td></tr>
<tr><td><code>items_per_zone</code></td><td>items ÷ storage zones</td><td>Average number of items per area visited</td></tr>
<tr><td><code>items_times_zones</code></td><td>items × storage zones</td><td>A combined measure of order size and areas visited</td></tr>
</tbody>
</table>

The dictionary controls which extra features are added. `True` includes a feature and `False` leaves it out. Models labelled `Original` use only the original seven features. `Selected extras` adds the enabled features to those seven. We start with volume and items multiplied by zones.

In [ ]:
# Turn an additional feature on with True or off with False.
feature_options = {
    "package_volume": True,
    "average_weight": False,
    "average_price": False,
    "fragile_share": False,
    "items_per_zone": False,
    "items_times_zones": True,
}

We want to compare models using the original information with models that also receive selected combinations. A transformer makes those calculations repeatable inside a pipeline: it receives a table and returns a copy with the chosen columns added. Its `fit` method does not need to estimate values here because the formulas are fixed; its `transform` method applies those formulas to each table it receives.

In Week 1, a class grouped data and methods. Here `AddOrderFeatures` follows the same idea, while also following the interface expected by scikit-learn. Its options are stored on the object; the pipeline calls its methods at the appropriate point. `fit_transform` combines fitting and transformation for the preview below. A transformer such as `StandardScaler`, unlike this fixed set of formulas, learns values during `fit`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# This scikit-learn compatible transformer adds the enabled columns.
class AddOrderFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, options=None):
        self.options = options

    # No quantities need to be learned for these formulas.
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Work on a copy so the original table stays unchanged.
        result = X.copy()
        options = self.options or {}

        # Each enabled option adds one column calculated from existing inputs.
        if options.get("package_volume", False):
            result["package_volume_cm3"] = X["package_side_cm"] ** 3
        if options.get("average_weight", False):
            result["average_weight"] = X["total_weight_kg"] / X["item_count"]
        if options.get("average_price", False):
            result["average_price"] = X["order_value_eur"] / X["item_count"]
        if options.get("fragile_share", False):
            result["fragile_share"] = X["fragile_count"] / X["item_count"]
        if options.get("items_per_zone", False):
            result["items_per_zone"] = X["item_count"] / X["storage_zones"]
        if options.get("items_times_zones", False):
            result["items_times_zones"] = X["item_count"] * X["storage_zones"]

        return result

# Preview the selected transformations on five training orders. fit_transform() is a convenience method that calls fit() and then transform().
AddOrderFeatures(feature_options).fit_transform(X_train.head())

<h2 id="section-7-pipelines-and-cross-validation">7. Pipelines and cross-validation</h2>

A pipeline applies steps in sequence: add features, scale, and fit a model. To fit a scaler means to calculate and store the mean and standard deviation of each feature. During cross-validation, each fold calculates these values using **only that fold's training rows**, then uses the stored values to transform both its training and validation rows. This keeps the validation orders separate when we evaluate the model on data it has not seen.

In week 3 we used one fixed validation set to compare choices. Here, **cross-validation** makes several comparisons within the training data. We divide those data into three parts, called **folds**:

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th>Run</th><th>Used to fit the pipeline</th><th>Used to validate it</th></tr></thead>
<tbody>
<tr><td>1</td><td>Folds 2 and 3</td><td>Fold 1</td></tr>
<tr><td>2</td><td>Folds 1 and 3</td><td>Fold 2</td></tr>
<tr><td>3</td><td>Folds 1 and 2</td><td>Fold 3</td></tr>
</tbody>
</table>

Each run starts with a fresh model and scaler. Its validation fold represents unseen orders for that run; the separate test set is still untouched. **Stratified** means the folds are arranged to have approximately similar proportions of the two classes. We compare the three AUC values and their mean, rather than relying on one particular split.

Our first model is **logistic regression**. Despite its name, it can classify orders: it combines the input features and produces an estimated probability for class 1. It is a useful simpler model to compare with a neural network. Here, a constant score for every order provides an even simpler reference: its AUC is 0.5.

The pipeline applies the same preparation to the training and validation orders in each run. It first adds the selected features, then learns the scaler from the fitting rows, then trains the model. Validation rows are transformed with the already learned scaler and used to calculate AUC. After selection, the final pipeline is fitted on all development rows and its stored scaler can transform one or many new orders.

**Self-check:** If validation weights are much larger than training weights, should their values change the mean used by the scaler? No. Using validation values to calculate that mean would let information about the validation set influence the training process.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Reuse the same reproducible stratified folds for all comparisons.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Build the same sequence for either model; passthrough skips scaling.
def make_pipeline(model, options=None, scale=True):
    return Pipeline([
        ("features", AddOrderFeatures(options)),
        ("scaler", StandardScaler() if scale else "passthrough"),
        ("model", model),
    ])

# Evaluate logistic regression with original features on the training folds.
# Logistic regression is a model used for binary classification that predicts the probability of a positive outcome.
baseline = make_pipeline(LogisticRegression(max_iter=2000, random_state=42))
scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="roc_auc")

print("Fold AUCs:", scores.round(3))
print(f"Mean: {scores.mean():.3f}; fold standard deviation: {scores.std():.3f}")

**Self-check:** Compare the three fold AUCs above with the constant-score reference of 0.5. Does the model rank long and short orders better than a score that is the same for every order? Are the fold results similar, or do they vary?

The fold standard deviation summarizes how much these three AUC values differ from their mean. A smaller value means these particular fold results are closer together. Three folds alone do not tell us exactly how the model will perform on every possible new dataset.

<h2 id="section-8-a-small-neural-network">8. A small neural network</h2>

The network receives the features, processes them through one or more **hidden layers**, then outputs a score for class 1. Each layer contains units called neurons. The connections between them have numbers called **weights**; training adjusts these weights to make useful predictions. Additional layers allow the network to build combinations of the patterns found by earlier layers. A larger network is not guaranteed to predict better.

A model can fit peculiarities of its training orders that do not hold for new orders. To discourage this, we can add a penalty for large weights during training. This is called **L2 regularization**. The setting `alpha` controls the penalty strength: a larger value penalizes large weights more strongly. Too much penalty can also make the model too inflexible, so we compare settings using cross-validation.

Our first example has one hidden layer with eight neurons, written `(8,)`. In the search below, `(8, 8)` means two layers of eight neurons and `(8, 8, 8)` means three. A convergence warning means that training stopped without meeting its stopping criterion; investigate such warnings when interpreting results.

For example, a network might assign a large positive weight to one input and a large negative weight to another to fit small details of the training observations. The penalty discourages large weights, though it cannot ensure the resulting model will work well on new data. The cross-validation comparison helps us examine that question.

In [ ]:
# One hidden layer with 8 neurons; alpha sets the penalty on large weights.
network = make_pipeline(MLPClassifier(hidden_layer_sizes=(8,), alpha=0.01,
                                      max_iter=1000, random_state=42))
network_scores = cross_val_score(network, X_train, y_train, cv=cv, scoring="roc_auc")

print("Network fold AUCs:", network_scores.round(3))
print(f"Mean: {network_scores.mean():.3f}; fold standard deviation: {network_scores.std():.3f}")

<h2 id="section-9-search-hyperparameters-and-compare-four-candidates">9. Search hyperparameters and compare four candidates</h2>

During training a model learns **parameters**, such as the weights of its connections. **Hyperparameters** are settings chosen before training, such as the number of hidden layers. Different settings may work better or worse for new orders.

`GridSearchCV` automates the comparison: for each candidate setting it trains and validates a fresh pipeline on each fold, averages the validation AUC values, and chooses the setting with the highest mean. It then fits that setting again on all training orders. This does not use the separate test set. The double underscore in `model__C` connects a setting to the pipeline step named `model`.

For logistic regression, `C` controls the strength of a penalty on model weights: a smaller `C` means stronger regularization. For the neural network, a larger `alpha` means stronger regularization. Regularization encourages smaller weights; it does not guarantee better predictions. We try two values of each setting and one, two or three hidden layers for the network.

The two models use appropriate, different grids. Each model is evaluated with original features and with the selected additions. ROC-AUC uses scores, not hard class predictions. In the results table, `neurons_per_layer` lists the number of neurons in each hidden layer, so `8 → 8` means two layers with eight neurons each. `C` applies to logistic regression; `alpha` and `neurons_per_layer` apply to the neural network.

For example, the network grid has three choices for its hidden-layer sizes and two choices for `alpha`: `3 × 2 = 6` combinations. With three validation folds, that is 18 fits for one feature set, followed by a refit of its best setting. We repeat the comparison with and without the selected extra features. The table summarizes each search, rather than listing all intermediate fits.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define one small search grid per model; model__ refers to the pipeline step.
model_specs = {
    "Logistic regression": (LogisticRegression(max_iter=2000, random_state=42),
                            {"model__C": [0.1, 1.0]}),
    "Neural network": (MLPClassifier(max_iter=1000, random_state=42),
                       {"model__hidden_layer_sizes": [(8,), (8, 8), (8, 8, 8)],
                        "model__alpha": [0.01, 0.1]}),
}

# Keep the fitted searches and their summary rows for later comparison.
searches = {}
rows = []

# For each model, search once with original and once with additional features.
for model_name, (model, grid) in model_specs.items():
    for feature_set, options in [("Original", None), ("Selected extras", feature_options)]:
        search = GridSearchCV(make_pipeline(model, options), grid, cv=cv,
                              scoring="roc_auc", refit=True, n_jobs=1)
        # Fit and compare settings using only the training data and CV folds.
        search.fit(X_train, y_train)

        key = (model_name, feature_set)
        searches[key] = search
        best = search.best_index_
        # Put each setting in its own column so the table remains readable.
        params = search.best_params_
        layers = params.get("model__hidden_layer_sizes")
        rows.append({"model": model_name, "features": feature_set,
                     "mean_auc": search.best_score_,
                     "fold_std": search.cv_results_["std_test_score"][best],
                     "C": params.get("model__C", None),
                     "alpha": params.get("model__alpha", None),
                     "neurons_per_layer": "–" if layers is None else " → ".join(str(size) for size in layers)})

# Show the strongest cross-validation mean first.
results = pd.DataFrame(rows).sort_values("mean_auc", ascending=False)
results

Read `mean_auc` as the average of the three validation AUCs and `fold_std` as a summary of how much those three values differ. The listed settings are the best of the candidates we tried, not necessarily the best settings possible.

Some settings can look better partly because of the particular orders in these folds. When we pick the largest of several estimates, it may be optimistic. A tiny difference alone does not establish a universal winner. Consider model simplicity, variation across folds and the fact that these data are synthetic.

**Self-check:** Did the engineered features help each model? Does an improvement establish that the new feature caused packing time to change?

<h2 id="section-10-investigate-scaling">10. Investigate scaling</h2>

We compare scaling on and off for the same default model settings and selected extra features, using the same folds. This is a controlled illustration, not another hyperparameter grid. The values need not match the tuned results above. Training may reach its iteration limit, especially without scaling; the corresponding AUC should be interpreted with care. If you use this comparison to choose a final approach, it becomes part of model selection.

Within one model, compare the rows labelled `True` and `False` in the `scaled` column. An AUC difference describes the result for these folds and settings; it does not isolate scaling from every possible training difficulty or establish a general rule. A convergence warning indicates that a model stopped before reaching its stopping criterion. The long warnings are hidden in this demonstration, so the table alone does not identify which fits produced them.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Repeat each default model on the same folds with and without scaling.
scale_rows = []

# Hide only convergence warnings inside this block, keeping the output readable.
# These warnings mean training reached a limit before its stopping criterion.
# Hiding them does not fix the issue; unscaled results need particular care.
# Other warning types remain visible, and the filter ends with this block.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)

    for name, (model, _) in model_specs.items():
        for scale in [True, False]:
            pipe = make_pipeline(model, feature_options, scale=scale)
            fold_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
            scale_rows.append({"model": name, "scaled": scale,
                               "mean_auc": fold_scores.mean(), "fold_std": fold_scores.std()})

pd.DataFrame(scale_rows)

<h2 id="section-11-select-and-evaluate-once-on-the-test-set">11. Select and evaluate once on the test set</h2>

For this worked example we select the highest mean cross-validation AUC among the four **scaled searches**. For nearly tied results, a simpler model might be a defensible alternative; here we use a reproducible rule. This automatically chosen rule does not select the unscaled demonstration above. The selected search has already refitted its best pipeline on all training rows (training + validation). We now use the previously untouched test set once.

The final fit has its own scaler learned from all development orders. It applies those stored values to the test orders before producing their scores. A later new order would be transformed with the same stored values; the scaler and model would need to be retrained together if those values were changed.

In [ ]:
# Select the top cross-validation result; the test set did not guide this choice.
winner_row = results.iloc[0]
winner_key = (winner_row["model"], winner_row["features"])
winner = searches[winner_key]
print("Selected:", winner_key, winner.best_params_)

# Column 1 contains the predicted score for class 1 (over 15 minutes).
test_scores = winner.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_scores)
test_fpr, test_tpr, _ = roc_curve(y_test, test_scores)
print(f"Held-out test ROC-AUC: {test_auc:.3f}")

plt.plot(test_fpr, test_tpr, label=f"Selected model (AUC {test_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Constant-score reference")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC on held-out test orders")
plt.legend()
plt.show()

The test AUC describes how well the chosen model ranks these held-out orders. It gives an estimate for new orders from the same simulated process; another sample of orders could give a different result. It does not establish performance in a real warehouse. If we repeatedly use this test result to choose features or models, our choices start depending on these orders and the test no longer provides the same independent check.

<h2 id="section-12-recap-and-outlook">12. Recap and outlook</h2>

Answer in your own words:

1. What is the prediction question, and which information is available at prediction time?
2. What is the constant-score baseline and what does ROC-AUC measure?
3. Why must scaling be fitted within each cross-validation fold?
4. Did the additional features or the neural network provide a meaningful advantage? What uncertainty remains?
5. What would you need before using such predictions in a real warehouse?

In a real data project, also consider time ordering, repeated customers or other related observations when choosing a split. The independent simulated orders here do not prove that a random split is suitable in another setting.

A strong AUC in this exercise shows that the model ranks these simulated orders well under the stated split. It does not show that the inputs cause long packing times or that the same pattern holds in another warehouse. The generator deliberately constructs this teaching dataset, so its results are a useful demonstration rather than evidence about a business.

Week 5 will use the knowledge gathered so far and apply it to a portfolio optimization problem. 

<h2 id="section-optional-your-turn">Optional: Your turn</h2>

Choose other options in `feature_options`, explain your expectations, and rerun the feature comparison on the **training data**. Consider average price: it may predict packing time indirectly through related order characteristics even though price itself need not cause packing time. If you have already inspected the test result above, treat further experiments as exploratory; a fresh independent test set would be needed for an unbiased final assessment of a newly selected approach.

For a second exploration, compare a different threshold on the small illustrative example and explain the resulting trade-off between false positives and false negatives.

### Further optional investigations

Choose one or more investigations. State your expectation first, work with training data and fresh model objects, then explain your findings and limitations. The final test set has already been inspected, so a newly selected approach needs a new independent test for final assessment.

1. **Compare feature choices.** Pick two different combinations from the dictionary. Explain why each might help, compare their cross-validation AUCs on the same folds, and decide whether the difference would justify collecting those inputs.
2. **Examine individual decisions.** Using the six-order example, compare two thresholds. Count each type of mistake and explain which threshold you would choose if missing a long order were more costly than unnecessarily preparing for a short one. What extra information would make the decision more realistic?
3. **Investigate fold sensitivity.** Keep the original test set aside. Repeat a fixed model comparison using several fixed cross-validation seeds on development data. Report the range of mean AUCs and whether the ordering changes; do not select a seed to obtain a preferred result.

In [ ]:
# Optional exploration workspace
# Start by copying data or recreating a generator so the walkthrough objects remain unchanged.

### Before you finish

1. Restart the kernel and run all cells from top to bottom.
2. Revisit any self-check whose answer was surprising.
3. Save the notebook. If you are working in Binder, download your notebook before closing the session.